In [ ]:
import sys
from pathlib import Path

# Pointer sur root de project
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

# Démarer experment mlflow
import config.mlflow

from src.model import train_model, predict
from src.spliting import split_dataset
from src.dataset import load_and_validate_yolo_dataset, analyze_image_quality, find_duplicate_images
from src.yolo_dataset import prepare_yolo_dataset, validate_yolo_dataset, create_data_yaml, analyze_yolo_annotations
from src.visualization import visualization_results_csv, analyze_class_distribution, draw_yolo_samples
from src.data_augmentations import visualize_horizontal_flip
from src.const import PROJECT_ROOT, DATASET_PATH, YOLO_DATASET_PATH, EPOCHS, SIZE_IMAGE, BATCH_SIZE, PATIENCE, REGISTER_MODEL_NAME
from src.mlflow import set_model_alias, load_model

#### Découverte du dataset

In [ ]:
dataset = load_and_validate_yolo_dataset(DATASET_PATH)

# Afficher la distribution
print(f"{'Split':<15} {'Classe':>10} {'Samples valides':>15}")
print("-" * 42)

for group in dataset:
    print(f"{group['split']:<15} {group['class']:>10} {len(group['samples']):>14}")

print("-" * 42)

### Visualisation imageswith bounding box

In [ ]:
# pass just 12 sample for drow yolo
draw_yolo_samples(
    [
        sample
        for group in dataset
        for sample in group["samples"]
    ][:12]
)

#### Analyse de la distribution des classes

In [ ]:
analyze_class_distribution(dataset)

#### Vérification des dimensions et de la qualité des images

In [ ]:
analyze_image_quality(dataset)

#### Vérifier les doublons d’images

In [ ]:
find_duplicate_images(dataset)

#### Data spliting

In [ ]:
train, val, test = split_dataset(dataset)

##### Préparer la structure du dataset pour YOLOv8

In [ ]:
prepare_yolo_dataset({
    "train": train,
    "val": val,
    "test": test,
})

#### Créer le fichier data.yaml

In [ ]:
create_data_yaml()

#### Charger et vérifier le dataset avec Ultralytics

In [ ]:
validate_yolo_dataset()

#### Analyser la distribution des classes dans les Bounding Boxes

In [ ]:
analyze_yolo_annotations(YOLO_DATASET_PATH)

##### Visualiser les Data Augmentations.

In [ ]:
visualize_horizontal_flip(train[0])

#### Préparer le premier entraînement

In [ ]:
# Lancer l'entraînement sur yolov8n et 50 epochs et 10 patience
train_model(
    base_model='yolov8n.pt',
    epochs=EPOCHS,
    image_size=SIZE_IMAGE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    run_dir=PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_nano'
)
visualization_results_csv(PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_nano' / 'results.csv')

# Lancer l'entraînement sur yolov8n et 100 epochs et 20 patience
train_model(
    base_model='yolov8n.pt',
    epochs=EPOCHS + 50,
    image_size=SIZE_IMAGE,
    batch=BATCH_SIZE,
    patience=PATIENCE + 10,
    run_dir=PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_nano_2'
)
visualization_results_csv(PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_nano_2' / 'results.csv')

# Lancer l'entraînement sur yolov8s et 50 epochs et 10 patience
train_model(
    base_model='yolov8s.pt',
    epochs=EPOCHS,
    image_size=SIZE_IMAGE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    run_dir=PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_small'
)
visualization_results_csv(PROJECT_ROOT / 'runs' / 'neurovision' / 'yolov8_small' / 'results.csv')

#### Créer l'alias champion.

In [ ]:
set_model_alias(
    name=REGISTER_MODEL_NAME,
    version=1
)

#### Loead baset model by alais champion

In [ ]:
# Charger le modèle champion
model_champion = load_model()

#### Visualiser les prédictions sur des IRM de test

In [ ]:
TEST_IMAGES_DIR = YOLO_DATASET_PATH / "images/test"

predict(
    model=model_champion,
    sources=[str(path) for path in TEST_IMAGES_DIR.iterdir()[:6]],
    run_dir=PROJECT_ROOT / 'runs' / 'neurovision' / 'test_predictions'
)

#### Sauvegarder et exporter le modèle YOLOv8

In [ ]:
# import shutil

# from ultralytics import YOLO

# # Source : meilleur modèle
# BEST_MODEL_PATH = Path(
#     "../runs/neurovision/yolov8n_baseline/weights/best.pt"
# )

# # Destination
# MODELS_DIR = Path("models")
# FINAL_MODEL_PATH = MODELS_DIR / "neurovision_yolov8n.pt"

# # Vérifier le modèle
# if not BEST_MODEL_PATH.exists():
#     raise FileNotFoundError(
#         f"Modèle introuvable : {BEST_MODEL_PATH}"
#     )

# # Créer le dossier models
# MODELS_DIR.mkdir(parents=True, exist_ok=True)

# # Copier le modèle
# shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

# # Vérifier la sauvegarde
# if FINAL_MODEL_PATH.exists():
#     print("Modèle sauvegardé avec succès !")
#     print(f"Chemin : {FINAL_MODEL_PATH.resolve()}")
#     print(f"Taille : {FINAL_MODEL_PATH.stat().st_size / (1024**2):.2f} MB")
    

# # Export ver onnx
# model = YOLO("models/neurovision_yolov8n.pt")

# onnx_path = model.export(
#     format="onnx",
#     imgsz=640
# )

# print(f"ONNX model exported: {onnx_path}")
